# 02. Preparacion de metadatos y espectrogramas HMS

Este notebook consume el manifiesto creado por `01_exploracion.ipynb`. Lee `data/train.csv` y los archivos `data/train_spectrograms/<spectrogram_id>.parquet`. Solo prepara el conjunto de desarrollo; no abre ni transforma el holdout interno ni los archivos `test` de Kaggle.

## Estrategia

- Validacion cruzada `StratifiedGroupKFold` con `patient_id` como grupo.
- Los seis conteos de votos se normalizan y se conservan como objetivo probabilistico.
- La clase mayoritaria derivada de los votos se usa solo para estratificar folds; `expert_consensus` queda como referencia descriptiva.
- Los Parquet de entrenamiento se consultan por identificador y se recorta la ventana indicada por el offset.
- Se normaliza cada imagen por separado y se replica a tres canales para un backbone de vision preentrenado, como EfficientNet-B0.
- No se ajustan escaladores globales sobre todo el dataset ni se usan datos de validacion para estimar transformaciones.

Ejecutar primero el notebook 01. Los artefactos ligeros de desarrollo se guardan en `data/processed/`, que no se versiona.

In [5]:
%pip install -q numpy pandas scikit-learn pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Carga del manifiesto y del conjunto de desarrollo

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

RANDOM_STATE = 42
N_SPLITS = 5
VOTE_COLUMNS = [
    "seizure_vote",
    "lpd_vote",
    "gpd_vote",
    "lrda_vote",
    "grda_vote",
    "other_vote",
]
CLASS_NAMES = [column[:-5].upper() for column in VOTE_COLUMNS]


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("No se encontro la raiz del proyecto con las carpetas data/ y notebooks/.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw" / "hms"
TRAIN_CSV_CANDIDATES = [
    DATA_DIR / "train.csv",
    RAW_DATA_DIR / "train.csv",
    DATA_DIR / "raw" / "train.csv",
]
TRAIN_CSV_PATH = next((path for path in TRAIN_CSV_CANDIDATES if path.is_file()), None)
SPECTROGRAM_DIR_CANDIDATES = [
    DATA_DIR / "train_spectrograms",
    RAW_DATA_DIR / "train_spectrograms",
    DATA_DIR / "raw" / "train_spectrograms",
]
SPECTROGRAM_DIR = next((path for path in SPECTROGRAM_DIR_CANDIDATES if path.is_dir()), None)
SPLIT_MANIFEST_PATH = DATA_DIR / "processed" / "split_manifest.json"
if TRAIN_CSV_PATH is None:
    raise FileNotFoundError("No se encontro train.csv en data/ ni en las rutas alternativas data/raw/.")
if not SPLIT_MANIFEST_PATH.is_file():
    raise FileNotFoundError("No existe split_manifest.json. Ejecute primero notebooks/01_exploracion.ipynb.")

with SPLIT_MANIFEST_PATH.open(encoding="utf-8") as manifest_file:
    split_manifest = json.load(manifest_file)
if split_manifest.get("group_column") != "patient_id":
    raise ValueError("El manifiesto no registra una separacion por patient_id.")

development_patient_ids = set(split_manifest["development_patient_ids"])
holdout_patient_ids = set(split_manifest["holdout_patient_ids"])
if development_patient_ids.intersection(holdout_patient_ids):
    raise ValueError("El manifiesto contiene pacientes en ambos conjuntos.")

required_columns = {
    "eeg_id",
    "eeg_sub_id",
    "eeg_label_offset_seconds",
    "spectrogram_id",
    "spectrogram_sub_id",
    "spectrogram_label_offset_seconds",
    "label_id",
    "patient_id",
    "expert_consensus",
    *VOTE_COLUMNS,
}
metadata = pd.read_csv(TRAIN_CSV_PATH, usecols=lambda column: column in required_columns)
missing_columns = required_columns.difference(metadata.columns)
if missing_columns:
    raise ValueError(f"Faltan columnas HMS requeridas: {sorted(missing_columns)}")
metadata["patient_id_key"] = metadata["patient_id"].astype(str)
development_metadata = metadata.loc[
    metadata["patient_id_key"].isin(development_patient_ids)
].copy().reset_index(drop=True)
del metadata

if development_metadata.empty:
    raise ValueError("La particion de desarrollo no contiene filas; verifique el manifiesto y train.csv.")
if not set(development_metadata["patient_id_key"].unique()).isdisjoint(holdout_patient_ids):
    raise AssertionError("La particion de desarrollo contiene pacientes del holdout.")

print(f"Archivo de etiquetas: {TRAIN_CSV_PATH.relative_to(PROJECT_ROOT)}")
print(f"Filas de desarrollo: {len(development_metadata):,}")
print(f"Pacientes de desarrollo: {development_metadata['patient_id_key'].nunique():,}")
print(f"Directorio de espectrogramas: {SPECTROGRAM_DIR.relative_to(PROJECT_ROOT) if SPECTROGRAM_DIR else 'no encontrado'}")
print("El holdout y los archivos de test no se cargan para preprocesamiento.")

Archivo de etiquetas: data\train.csv
Filas de desarrollo: 84,131
Pacientes de desarrollo: 1,560
Directorio de espectrogramas: data\train_spectrograms
El holdout y los archivos de test no se cargan para preprocesamiento.


## 2. Objetivos probabilisticos y validacion agrupada

Los votos se convierten en probabilidades por fila. La clase mayoritaria derivada de esos votos se usa solamente para estratificar la particion; el entrenamiento posterior debe usar el vector de probabilidades completo (por ejemplo, entropia cruzada con etiquetas blandas).

In [2]:
vote_matrix = development_metadata[VOTE_COLUMNS].astype(float)
if vote_matrix.isna().any().any():
    raise ValueError("Hay votos faltantes en desarrollo; no se deben interpretar como cero.")
if (vote_matrix < 0).any().any():
    raise ValueError("Los votos deben ser no negativos.")

vote_totals = vote_matrix.sum(axis=1)
valid_target_rows = vote_totals.gt(0)
if not valid_target_rows.all():
    print(f"Se excluyen {int((~valid_target_rows).sum()):,} filas sin votos validos.")
    development_metadata = development_metadata.loc[valid_target_rows].copy().reset_index(drop=True)
    vote_matrix = development_metadata[VOTE_COLUMNS].astype(float)
    vote_totals = vote_matrix.sum(axis=1)

soft_targets = vote_matrix.div(vote_totals, axis=0)
dominant_classes = vote_matrix.idxmax(axis=1).str.replace("_vote", "", regex=False)
groups = development_metadata["patient_id_key"]

if groups.nunique() < N_SPLITS:
    raise ValueError(f"Se requieren al menos {N_SPLITS} pacientes para la validacion agrupada.")

cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
fold_ids = np.full(len(development_metadata), -1, dtype=np.int8)
for fold_id, (train_indices, validation_indices) in enumerate(
    cv.split(np.zeros(len(development_metadata)), dominant_classes, groups)
):
    train_patients = set(groups.iloc[train_indices])
    validation_patients = set(groups.iloc[validation_indices])
    if not train_patients.isdisjoint(validation_patients):
        raise AssertionError(f"Pacientes compartidos entre train y fold {fold_id}.")
    fold_ids[validation_indices] = fold_id

if (fold_ids < 0).any():
    raise AssertionError("Algunas filas no quedaron asignadas a un fold de validacion.")

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
row_ids = np.arange(len(development_metadata))
target_columns = [f"target_{name.lower()}" for name in CLASS_NAMES]

prepared_metadata = development_metadata[
    ["eeg_id", "eeg_sub_id", "spectrogram_id", "spectrogram_sub_id", "spectrogram_label_offset_seconds", "patient_id"]
].copy()
prepared_metadata.insert(0, "row_id", row_ids)
prepared_metadata["fold_id"] = fold_ids
for column, target_column in zip(VOTE_COLUMNS, target_columns):
    prepared_metadata[target_column] = soft_targets[column].to_numpy(dtype=np.float32)

fold_assignments_path = PROCESSED_DATA_DIR / "cv_fold_assignments.csv"
targets_path = PROCESSED_DATA_DIR / "development_targets.csv"
prepared_metadata[["row_id", "eeg_id", "eeg_sub_id", "patient_id", "fold_id"]].to_csv(
    fold_assignments_path, index=False
)
prepared_metadata.to_csv(targets_path, index=False)

print(f"Folds de validacion: {N_SPLITS}")
print(f"Filas con objetivos validos: {len(prepared_metadata):,}")
print("Archivos escritos solo con desarrollo:")
print(f"- {fold_assignments_path.relative_to(PROJECT_ROOT)}")
print(f"- {targets_path.relative_to(PROJECT_ROOT)}")

Folds de validacion: 5
Filas con objetivos validos: 84,131
Archivos escritos solo con desarrollo:
- data\processed\cv_fold_assignments.csv
- data\processed\development_targets.csv


## 3. Representacion visual de los espectrogramas

HMS ya distribuye espectrogramas derivados del EEG, por lo que no se recalcula STFT a partir del EEG crudo en esta etapa. Se extrae la ventana de 50 segundos indicada por los metadatos, se transforma a escala logaritmica, se normaliza por imagen y se redimensiona. La salida RGB es compatible con un backbone de vision preentrenado; el ajuste del modelo se realiza en el notebook 03.

In [6]:
if SPECTROGRAM_DIR is None:
    raise FileNotFoundError("No se encontro la carpeta data/train_spectrograms con los Parquet de entrenamiento.")

SPECTROGRAM_FILES = {
    int(path.stem): path
    for path in SPECTROGRAM_DIR.glob("*.parquet")
    if path.stem.isdigit()
}
if not SPECTROGRAM_FILES:
    raise FileNotFoundError(f"No se encontraron archivos Parquet individuales en {SPECTROGRAM_DIR}.")

development_spectrogram_ids = set(development_metadata["spectrogram_id"].astype(int).unique())
missing_spectrogram_ids = development_spectrogram_ids.difference(SPECTROGRAM_FILES)
if missing_spectrogram_ids:
    preview = sorted(missing_spectrogram_ids)[:10]
    raise FileNotFoundError(
        f"Faltan {len(missing_spectrogram_ids):,} espectrogramas de desarrollo. Ejemplos: {preview}"
    )

sample_spectrogram_id, sample_spectrogram_path = next(iter(SPECTROGRAM_FILES.items()))
sample_spectrogram = pd.read_parquet(sample_spectrogram_path)
print(f"Espectrogramas de entrenamiento disponibles: {len(SPECTROGRAM_FILES):,}")
print(f"IDs de desarrollo cubiertos: {len(development_spectrogram_ids):,}")
print(f"Ejemplo {sample_spectrogram_id}: {sample_spectrogram.shape}; primeras columnas: {sample_spectrogram.columns[:8].tolist()}")


def resize_and_normalize_spectrogram(spectrogram: np.ndarray, image_size: int = 224) -> np.ndarray:
    values = np.asarray(spectrogram, dtype=np.float32)
    if values.ndim != 2 or min(values.shape) < 2:
        raise ValueError("El espectrograma debe ser una matriz frecuencia por tiempo de al menos 2x2.")
    values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
    values = np.log1p(np.clip(values, a_min=0.0, a_max=None))
    lower, upper = np.percentile(values, [1, 99])
    if upper <= lower:
        normalized = np.zeros_like(values)
    else:
        normalized = np.clip((values - lower) / (upper - lower), 0.0, 1.0)

    source_time = np.linspace(0.0, 1.0, normalized.shape[1])
    target_time = np.linspace(0.0, 1.0, image_size)
    resized_time = np.stack(
        [np.interp(target_time, source_time, row) for row in normalized], axis=0
    )
    source_frequency = np.linspace(0.0, 1.0, normalized.shape[0])
    target_frequency = np.linspace(0.0, 1.0, image_size)
    resized = np.stack(
        [
            np.interp(target_frequency, source_frequency, resized_time[:, column])
            for column in range(image_size)
        ],
        axis=1,
    )
    return np.repeat(resized[..., np.newaxis], 3, axis=2).astype(np.float32)


def load_spectrogram_image(
    spectrogram_id: int,
    label_offset_seconds: float,
    parquet_dir: Path = SPECTROGRAM_DIR,
    window_seconds: int = 50,
    image_size: int = 224,
) -> np.ndarray:
    if parquet_dir is None or not Path(parquet_dir).is_dir():
        raise FileNotFoundError("Se requiere la carpeta train_spectrograms de HMS.")
    if pd.isna(label_offset_seconds):
        raise ValueError("label_offset_seconds no puede estar vacio.")

    parquet_path = Path(parquet_dir) / f"{int(spectrogram_id)}.parquet"
    if not parquet_path.is_file():
        raise FileNotFoundError(f"No existe el espectrograma de entrenamiento: {parquet_path}")

    spectrogram_rows = pd.read_parquet(parquet_path)
    if "time" not in spectrogram_rows.columns:
        raise ValueError(f"El archivo {parquet_path.name} no contiene la columna temporal 'time'.")

    start = float(label_offset_seconds)
    stop = start + window_seconds
    segment = spectrogram_rows.loc[
        spectrogram_rows["time"].between(start, stop, inclusive="left")
    ].sort_values("time")
    frequency_columns = [column for column in segment.columns if column != "time"]
    if segment.empty or not frequency_columns:
        raise ValueError(
            f"No se encontro una ventana de {window_seconds}s para spectrogram_id={spectrogram_id} "
            f"en [{start}, {stop})."
        )
    matrix = segment[frequency_columns].apply(pd.to_numeric, errors="coerce").to_numpy().T
    return resize_and_normalize_spectrogram(matrix, image_size=image_size)


print("Los Parquet se consultan por espectrograma para mantener bajo el uso de memoria.")

Espectrogramas de entrenamiento disponibles: 11,138
IDs de desarrollo cubiertos: 8,740
Ejemplo 1000086677: (300, 401); primeras columnas: ['time', 'LL_0.59', 'LL_0.78', 'LL_0.98', 'LL_1.17', 'LL_1.37', 'LL_1.56', 'LL_1.76']
Los Parquet se consultan por espectrograma para mantener bajo el uso de memoria.


In [7]:
synthetic_spectrogram = np.arange(80, dtype=np.float32).reshape(8, 10)
synthetic_image = resize_and_normalize_spectrogram(synthetic_spectrogram, image_size=32)
assert synthetic_image.shape == (32, 32, 3)
assert np.isfinite(synthetic_image).all()
assert synthetic_image.min() >= 0.0 and synthetic_image.max() <= 1.0
print("Prueba local sintetica: correcta", synthetic_image.shape)

nonzero_offset_rows = development_metadata.loc[
    development_metadata["spectrogram_label_offset_seconds"].gt(0)
]
sample_row = nonzero_offset_rows.iloc[0]
real_image = load_spectrogram_image(
    spectrogram_id=int(sample_row["spectrogram_id"]),
    label_offset_seconds=float(sample_row["spectrogram_label_offset_seconds"]),
    image_size=64,
)
assert real_image.shape == (64, 64, 3)
assert np.isfinite(real_image).all()
assert real_image.min() >= 0.0 and real_image.max() <= 1.0
print(
    f"Prueba con HMS real: spectrogram_id={sample_row['spectrogram_id']}, "
    f"offset={sample_row['spectrogram_label_offset_seconds']}, "
    f"imagen={real_image.shape}, rango=[{real_image.min():.3f}, {real_image.max():.3f}]"
)

Prueba local sintetica: correcta (32, 32, 3)
Prueba con HMS real: spectrogram_id=353733, offset=6.0, imagen=(64, 64, 3), rango=[0.000, 1.000]


## 4. Contrato con modelado y prevencion de fuga

- `data/processed/development_targets.csv` contiene solo filas de desarrollo, metadatos necesarios, fold y seis objetivos blandos.
- `data/processed/cv_fold_assignments.csv` permite reconstruir los folds desde otro notebook/kernel.
- Para cada fold, ajustar cualquier normalizacion aprendida, ponderacion de clases o seleccion de hiperparametros usando unicamente las filas de entrenamiento de ese fold.
- La normalizacion por imagen definida aqui no estima parametros compartidos entre filas.
- No usar el holdout interno para elegir arquitectura, epocas, umbrales ni transformaciones. Evaluarlo una sola vez tras congelar el pipeline.
- Las imagenes tienen espectrogramas, no EEG crudo; las afirmaciones sobre filtros, montaje bipolar o espectrogramas calculados de nuevo requieren otro experimento y no se asumen aqui.

El modelo recomendado para la siguiente etapa es una CNN de vision preentrenada (por ejemplo EfficientNet-B0) con una cabeza de seis salidas probabilisticas. Las metricas y conclusiones se completaran en los notebooks 03 y 04.